# **Executive Summary: Comparative Pharmacovigilance & Predictive Signaling of GLP-1 Adjuvants (2023–2026)**

**1. Problem Statement**

The rapid global adoption of GLP-1 receptor agonists (Semaglutide) and dual GIP/GLP-1 agonists (Tirzepatide) has outpaced long-term musculoskeletal safety data. While clinical trials suggest a 15–25% loss in lean muscle mass, real-world signals of Sarcopenia and Muscle Atrophy are emerging in post-marketing surveillance. Furthermore, the 2025–2026 shift toward Oral Formulations (Wegovy Pill, Rybelsus, and the upcoming Orforglipron) introduces a "Formulation Gap," where the administrative burden and stability of oral delivery may exacerbate adverse event (AE) reporting compared to traditional injectables.

**2. Project Objectives**

- **Signal Detection:** Quantify the Reporting Odds Ratio (ROR) for muscle-related AEs across the incretin class.

- **Formulation Comparison:** Isolate the "Oral vs. Injectable" safety profile, specifically focusing on administrative errors and GI-driven muscle wasting.

- **Predictive Analytics:** Utilize Machine Learning (Random Forest/XGBoost) to identify patient phenotypes most at risk of "Serious" outcomes (hospitalization/disability).

- **Stability Assessment:** Evaluate the 2026 "Tirzepatide–B12 Adduct" impurity signal in compounded vs. branded formulations to mitigate regulatory risk.

**3. Strategic milestones**

**4. Industry-Relevant Implications**

- **Labeling & Regulatory Strategy:** If the ROR for Sarcopenia remains >2.0 in the 2025–2026 cohort, the FDA may mandate a Black Box Warning or specific "Resistance Training" guidance on the labels for high-dose oral GLP-1s.

- **Market Differentiation:** Evidence that Tirzepatide’s GIP-agonist component provides a "Muscle-Protective" effect compared to pure GLP-1s would be a massive competitive advantage in the 2026 obesity market.

- **Cost-Containment:** For payers (Medicare/Medicaid), identifying at-risk sarcopenic patients early can prevent the $900/member/year excess cost associated with muscle-wasting complications.

- **Stability Risks:** The identification of chemical adducts (B12-Tirzepatide) in the 2026 data serves as a critical warning for the company’s own future co-formulation pipelines.



In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import drive

drive.mount('/content/drive')
# SAVE AS PARQUET - This is the secret to 60k records!
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Semaglutide_2023-2025_MAX.parquet"

API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2025, 12, 31)

def fetch_60k_safe():
    current_date = START_DATE
    all_dfs = []

    # Using the broad Substance search for maximum volume
    search_query = 'patient.drug.openfda.substance_name:"SEMAGLUTIDE"'

    while current_date <= END_DATE:
        d_str = current_date.strftime("%Y%m%d")
        skip = 0

        while True:
            url = (f"https://api.fda.gov/drug/event.json?api_key={API_KEY}"
                   f"&search=receivedate:{d_str}+AND+{search_query}"
                   f"&limit=1000&skip={skip}")

            try:
                response = requests.get(url, timeout=25)
                if response.status_code == 200:
                    results = response.json().get('results', [])
                    if not results: break

                    # 2. Flatten and Keep Essential Columns only
                    df_chunk = pd.json_normalize(results)
                    cols = [c for c in df_chunk.columns if any(x in c for x in ['id', 'date', 'reaction', 'drug', 'seriousness'])]
                    df_chunk = df_chunk[cols]

                    all_dfs.append(df_chunk)
                    print(f"✅ {d_str}: Found {len(df_chunk)} records.")

                    if len(results) == 1000:
                        skip += 1000
                    else:
                        break
                else: break
            except: break
            time.sleep(0.1)

        # 3. Intermediate Save to Drive every 30 days to save RAM
        if current_date.day == 28 and all_dfs:
            temp_df = pd.concat(all_dfs, ignore_index=True)
            temp_df.to_parquet(SAVE_PATH, index=False)
            print(f"💾 RAM Cleared. Progress saved to Drive.")

        current_date += timedelta(days=1)

    # Final Save
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 SUCCESS! Total Records: {len(final_df)}")

fetch_60k_safe()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 20230101: Found 2 records.
✅ 20230102: Found 14 records.
✅ 20230103: Found 19 records.
✅ 20230104: Found 26 records.
✅ 20230105: Found 12 records.
✅ 20230106: Found 22 records.
✅ 20230107: Found 1 records.
✅ 20230108: Found 6 records.
✅ 20230109: Found 15 records.
✅ 20230110: Found 16 records.
✅ 20230111: Found 23 records.
✅ 20230112: Found 20 records.
✅ 20230113: Found 200 records.
✅ 20230114: Found 1 records.
✅ 20230116: Found 9 records.
✅ 20230117: Found 233 records.
✅ 20230118: Found 40 records.
✅ 20230119: Found 23 records.
✅ 20230120: Found 26 records.
✅ 20230121: Found 1 records.
✅ 20230122: Found 1 records.
✅ 20230123: Found 1000 records.
✅ 20230123: Found 549 records.
✅ 20230124: Found 16 records.
✅ 20230125: Found 24 records.
✅ 20230126: Found 9 records.
✅ 20230127: Found 18 records.
✅ 20230128: Found 4 records.
💾 RAM Cleared. Progress saved to Dr

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# UPDATE PATH FOR TIRZEPATIDE
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Tirzepatide_2023_2025_MAX.parquet"

API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
# START FROM 2023 to capture the pre-Zepbound baseline
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2025, 12, 31)

def fetch_tirzepatide_data():
    current_date = START_DATE
    all_dfs = []

    # Target the active substance to catch Mounjaro AND Zepbound
    search_query = 'patient.drug.openfda.substance_name:"TIRZEPATIDE"'

    while current_date <= END_DATE:
        d_str = current_date.strftime("%Y%m%d")
        skip = 0

        while True:
            url = (f"https://api.fda.gov/drug/event.json?api_key={API_KEY}"
                   f"&search=receivedate:{d_str}+AND+ {search_query}"
                   f"&limit=1000&skip={skip}")

            try:
                response = requests.get(url, timeout=25)
                if response.status_code == 200:
                    results = response.json().get('results', [])
                    if not results: break

                    df_chunk = pd.json_normalize(results)
                    # Filter for essential columns to keep file size small
                    cols = [c for c in df_chunk.columns if any(x in c for x in ['id', 'date', 'reaction', 'drug', 'seriousness'])]
                    df_chunk = df_chunk[cols]

                    all_dfs.append(df_chunk)
                    print(f"✅ {d_str}: Found {len(df_chunk)} Tirzepatide records.")

                    if len(results) == 1000: skip += 1000
                    else: break
                else: break
            except: break
            time.sleep(0.1)

        # Periodic save to Drive
        if current_date.day == 28 and all_dfs:
            pd.concat(all_dfs, ignore_index=True).to_parquet(SAVE_PATH, index=False)
            print(f"💾 Progress saved for {current_date.strftime('%B %Y')}")

        current_date += timedelta(days=1)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 SUCCESS! Total Tirzepatide Records: {len(final_df)}")

fetch_tirzepatide_data()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 20230101: Found 13 Tirzepatide records.
✅ 20230102: Found 38 Tirzepatide records.
✅ 20230103: Found 48 Tirzepatide records.
✅ 20230104: Found 45 Tirzepatide records.
✅ 20230105: Found 55 Tirzepatide records.
✅ 20230106: Found 28 Tirzepatide records.
✅ 20230107: Found 51 Tirzepatide records.
✅ 20230108: Found 39 Tirzepatide records.
✅ 20230109: Found 45 Tirzepatide records.
✅ 20230110: Found 56 Tirzepatide records.
✅ 20230111: Found 59 Tirzepatide records.
✅ 20230112: Found 58 Tirzepatide records.
✅ 20230113: Found 51 Tirzepatide records.
✅ 20230114: Found 20 Tirzepatide records.
✅ 20230115: Found 26 Tirzepatide records.
✅ 20230116: Found 47 Tirzepatide records.
✅ 20230117: Found 67 Tirzepatide records.
✅ 20230118: Found 72 Tirzepatide records.
✅ 20230119: Found 51 Tirzepatide records.
✅ 20230120: Found 47 Tirzepatide records.
✅ 20230121: Found 21 Tirzepatid

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# UPDATE PATH FOR LIRAGLUTIDE
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Liraglutide_2023_2025_MAX.parquet"

API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2025, 12, 31)

def fetch_liraglutide_data():
    current_date = START_DATE
    all_dfs = []

    # Substance search captures Brand + Generics + Compounders
    search_query = 'patient.drug.openfda.substance_name:"LIRAGLUTIDE"'

    while current_date <= END_DATE:
        d_str = current_date.strftime("%Y%m%d")
        skip = 0

        while True:
            url = (f"https://api.fda.gov/drug/event.json?api_key={API_KEY}"
                   f"&search=receivedate:{d_str}+AND+{search_query}"
                   f"&limit=1000&skip={skip}")

            try:
                response = requests.get(url, timeout=25)
                if response.status_code == 200:
                    results = response.json().get('results', [])
                    if not results: break

                    df_chunk = pd.json_normalize(results)
                    # Use same essential columns as your other sets for easy merging later
                    cols = [c for c in df_chunk.columns if any(x in c for x in ['id', 'date', 'reaction', 'drug', 'seriousness'])]
                    df_chunk = df_chunk[cols]

                    all_dfs.append(df_chunk)
                    print(f"✅ {d_str}: Found {len(df_chunk)} Liraglutide records.")

                    if len(results) == 1000: skip += 1000
                    else: break
                else: break
            except: break
            time.sleep(0.1)

        if current_date.day == 28 and all_dfs:
            pd.concat(all_dfs, ignore_index=True).to_parquet(SAVE_PATH, index=False)
            print(f"💾 RAM Cleared. Monthly progress saved.")

        current_date += timedelta(days=1)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 SUCCESS! Total Liraglutide Records: {len(final_df)}")

fetch_liraglutide_data()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 20230102: Found 6 Liraglutide records.
✅ 20230103: Found 2 Liraglutide records.
✅ 20230104: Found 8 Liraglutide records.
✅ 20230105: Found 2 Liraglutide records.
✅ 20230106: Found 8 Liraglutide records.
✅ 20230108: Found 1 Liraglutide records.
✅ 20230109: Found 2 Liraglutide records.
✅ 20230110: Found 4 Liraglutide records.
✅ 20230111: Found 7 Liraglutide records.
✅ 20230112: Found 4 Liraglutide records.
✅ 20230113: Found 12 Liraglutide records.
✅ 20230114: Found 1 Liraglutide records.
✅ 20230115: Found 1 Liraglutide records.
✅ 20230116: Found 4 Liraglutide records.
✅ 20230117: Found 3 Liraglutide records.
✅ 20230118: Found 5 Liraglutide records.
✅ 20230119: Found 8 Liraglutide records.
✅ 20230120: Found 10 Liraglutide records.
✅ 20230121: Found 1 Liraglutide records.
✅ 20230122: Found 1 Liraglutide records.
✅ 20230123: Found 16 Liraglutide records.
✅ 20230

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# PATH FOR CONTRAVE (Non-GLP Control)
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Contrave_2014_2025_MAX.parquet"

API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
START_DATE = datetime(2014, 1, 1)
END_DATE = datetime(2025, 12, 31)

def fetch_contrave_data():
    current_date = START_DATE
    all_dfs = []

    # Logic: Find reports with BOTH substances or the Brand Name
    # This filters out single-drug users (e.g., just Bupropion for smoking cessation)
    search_query = ('(patient.drug.openfda.substance_name:"NALTREXONE" '
                    'AND patient.drug.openfda.substance_name:"BUPROPION") '
                    'OR patient.drug.brand_name:"CONTRAVE"')

    while current_date <= END_DATE:
        d_str = current_date.strftime("%Y%m%d")
        skip = 0

        while True:
            url = (f"https://api.fda.gov/drug/event.json?api_key={API_KEY}"
                   f"&search=receivedate:{d_str}+AND+{search_query}"
                   f"&limit=1000&skip={skip}")

            try:
                response = requests.get(url, timeout=25)
                if response.status_code == 200:
                    results = response.json().get('results', [])
                    if not results: break

                    df_chunk = pd.json_normalize(results)
                    # Keep same columns as GLP-1 sets for easy merging
                    cols = [c for c in df_chunk.columns if any(x in c for x in ['id', 'date', 'reaction', 'drug', 'seriousness'])]
                    df_chunk = df_chunk[cols]

                    all_dfs.append(df_chunk)
                    print(f"✅ {d_str}: Found {len(df_chunk)} Contrave-related records.")

                    if len(results) == 1000: skip += 1000
                    else: break
                else: break
            except: break
            time.sleep(0.1)

        # Save progress
        if current_date.day == 28 and all_dfs:
            pd.concat(all_dfs, ignore_index=True).to_parquet(SAVE_PATH, index=False)

        current_date += timedelta(days=1)

    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 SUCCESS! Total Contrave Records: {len(final_df)}")

fetch_contrave_data()

Mounted at /content/drive
✅ 20140206: Found 1 Contrave-related records.
✅ 20140221: Found 1 Contrave-related records.
✅ 20141112: Found 1 Contrave-related records.
✅ 20141208: Found 3 Contrave-related records.
✅ 20141210: Found 1 Contrave-related records.
✅ 20141217: Found 2 Contrave-related records.
✅ 20141229: Found 2 Contrave-related records.
✅ 20141230: Found 1 Contrave-related records.
✅ 20141231: Found 2 Contrave-related records.
✅ 20150106: Found 1 Contrave-related records.
✅ 20150107: Found 66 Contrave-related records.
✅ 20150108: Found 1 Contrave-related records.
✅ 20150113: Found 1 Contrave-related records.
✅ 20150115: Found 1 Contrave-related records.
✅ 20150120: Found 2 Contrave-related records.
✅ 20150121: Found 1 Contrave-related records.
✅ 20150123: Found 2 Contrave-related records.
✅ 20150126: Found 1 Contrave-related records.
✅ 20150128: Found 1 Contrave-related records.
✅ 20150204: Found 1 Contrave-related records.
✅ 20150206: Found 1 Contrave-related records.
✅ 20150

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

# --- CONFIGURATION ---
API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4" # Use your actual key
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Contrave_Muscle_Signal_2023_2025.parquet"

# Define the "Muscle Signal" MedDRA Bucket
# These are the terms a doctor would use in 2026 to describe the wasting you are studying
MUSCLE_TERMS = [
    'MUSCLE ATROPHY',
    'SARCOPENIA',
    'MUSCULAR WEAKNESS',
    'MOBILITY DECREASED',
    'WEIGHT DECREASED',
    'GAIT DISTURBANCE'
]

def fetch_contrave_signals():
    # 1. Broad Search Query: Brand name OR Generic Combo
    # We use .exact for the MedDRA terms to ensure we get the full clinical phrase
    drug_query = '(patient.drug.brand_name:"CONTRAVE" OR (patient.drug.openfda.substance_name:"NALTREXONE HYDROCHLORIDE" AND patient.drug.openfda.substance_name:"BUPROPION HYDROCHLORIDE"))'

    # Create the reaction sub-query
    reaction_query = " OR ".join([f'patient.reaction.reactionmeddrapt.exact:"{term}"' for term in MUSCLE_TERMS])

    # Full API Search String
    full_search = f'{drug_query} AND ({reaction_query}) AND receivedate:[20230101 TO 20251231]'

    print(f"🚀 Initializing search for Contrave muscle signals...")

    base_url = f"https://api.fda.gov/drug/event.json?api_key={API_KEY}&search={full_search}"

    # Check total count first to verify the "Net" is working
    try:
        count_resp = requests.get(f"{base_url}&limit=1")
        total_records = count_resp.json().get('meta', {}).get('results', {}).get('total', 0)
        print(f"📊 Found {total_records} matching reports for the specified muscle terms.")

        if total_records == 0:
            print("💡 Insight: No signal found. This confirms Contrave as a strong 'Negative Control'.")
            return
    except Exception as e:
        print(f"❌ Error connecting to API: {e}")
        return

    # 2. Pagination Loop (if signals are found)
    all_results = []
    skip = 0
    while skip < total_records:
        url = f"{base_url}&limit=1000&skip={skip}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json().get('results', [])
            all_results.extend(data)
            print(f"✅ Downloaded {len(all_results)} / {total_records}...")
            skip += 1000
            time.sleep(0.2) # API Rate Limit protection
        else:
            break

    # 3. Process and Save
    df = pd.json_normalize(all_results)
    df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 Final Dataset Saved: {len(df)} records.")

fetch_contrave_signals()

🚀 Initializing search for Contrave muscle signals...
📊 Found 36 matching reports for the specified muscle terms.
✅ Downloaded 36 / 36...
🎉 Final Dataset Saved: 36 records.


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION ---
API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4" # Use your actual key
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Contrave_Muscle_Signal_2014_2022.parquet"

# Define the "Muscle Signal" MedDRA Bucket
# These are the terms a doctor would use in 2026 to describe the wasting you are studying
MUSCLE_TERMS = [
    'MUSCLE ATROPHY',
    'SARCOPENIA',
    'MUSCULAR WEAKNESS',
    'MOBILITY DECREASED',
    'WEIGHT DECREASED',
    'GAIT DISTURBANCE'
]

def fetch_contrave_signals():
    # 1. Broad Search Query: Brand name OR Generic Combo
    # We use .exact for the MedDRA terms to ensure we get the full clinical phrase
    drug_query = '(patient.drug.brand_name:"CONTRAVE" OR (patient.drug.openfda.substance_name:"NALTREXONE HYDROCHLORIDE" AND patient.drug.openfda.substance_name:"BUPROPION HYDROCHLORIDE"))'

    # Create the reaction sub-query
    reaction_query = " OR ".join([f'patient.reaction.reactionmeddrapt.exact:"{term}"' for term in MUSCLE_TERMS])

    # Full API Search String
    full_search = f'{drug_query} AND ({reaction_query}) AND receivedate:[20140101 TO 20221231]'

    print(f"🚀 Initializing search for Contrave muscle signals...")

    base_url = f"https://api.fda.gov/drug/event.json?api_key={API_KEY}&search={full_search}"

    # Check total count first to verify the "Net" is working
    try:
        count_resp = requests.get(f"{base_url}&limit=1")
        total_records = count_resp.json().get('meta', {}).get('results', {}).get('total', 0)
        print(f"📊 Found {total_records} matching reports for the specified muscle terms.")

        if total_records == 0:
            print("💡 Insight: No signal found. This confirms Contrave as a strong 'Negative Control'.")
            return
    except Exception as e:
        print(f"❌ Error connecting to API: {e}")
        return

    # 2. Pagination Loop (if signals are found)
    all_results = []
    skip = 0
    while skip < total_records:
        url = f"{base_url}&limit=1000&skip={skip}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json().get('results', [])
            all_results.extend(data)
            print(f"✅ Downloaded {len(all_results)} / {total_records}...")
            skip += 1000
            time.sleep(0.2) # API Rate Limit protection
        else:
            break

    # 3. Process and Save
    df = pd.json_normalize(all_results)
    df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 Final Dataset Saved: {len(df)} records.")

fetch_contrave_signals()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Initializing search for Contrave muscle signals...
📊 Found 72 matching reports for the specified muscle terms.
✅ Downloaded 72 / 72...
🎉 Final Dataset Saved: 72 records.


In [ ]:
import requests
import pandas as pd
import time

# --- CONFIGURATION ---
API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Contrave_GI_Signals_2023_2025.parquet"

# MedDRA Terms: Common vs. Serious GI Injury
GI_TERMS = [
    'NAUSEA', 'VOMITING', 'DIARRHOEA','CONSTIPATION', # Acute/Tolerability
    'GASTROPARESIS', 'INTESTINAL OBSTRUCTION',      # Serious/Structural
    'PANCREATITIS', 'GASTRIC STASIS', 'ILEUS'       # Serious/Functional
]

def fetch_contrave_gi():
    # Drug Query (Brand OR Generic Combo)
    drug_query = '(patient.drug.brand_name:"CONTRAVE" OR (patient.drug.openfda.substance_name:"NALTREXONE HYDROCHLORIDE" AND patient.drug.openfda.substance_name:"BUPROPION HYDROCHLORIDE"))'

    # Reaction Query (Standardized MedDRA PTs)
    reaction_query = " OR ".join([f'patient.reaction.reactionmeddrapt.exact:"{term}"' for term in GI_TERMS])

    # Combined Search String for 2023-2025
    full_search = f'{drug_query} AND ({reaction_query}) AND receivedate:[20230101 TO 20251231]'

    print(f"🚀 Scanning for GI signals in Contrave cohort...")

    base_url = f"https://api.fda.gov/drug/event.json?api_key={API_KEY}&search={full_search}"

    try:
        count_resp = requests.get(f"{base_url}&limit=1")
        total_records = count_resp.json().get('meta', {}).get('results', {}).get('total', 0)
        print(f"📊 Found {total_records} GI-related reports for Contrave.")

        if total_records == 0:
            print("💡 Observation: No GI signals found. Contrave appears highly tolerated in this dataset.")
            return
    except Exception as e:
        print(f"❌ Connection Error: {e}")
        return

    all_results = []
    skip = 0
    while skip < total_records:
        url = f"{base_url}&limit=1000&skip={skip}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json().get('results', [])
            all_results.extend(data)
            print(f"✅ Progress: {len(all_results)} / {total_records}")
            skip += 1000
            time.sleep(0.2)
        else:
            break

    # Convert to DataFrame and Save
    df = pd.json_normalize(all_results)

    # Tag these records for later ML classification
    df['is_control_group'] = True
    df['drug_class'] = 'CNS_WEIGHT_LOSS'

    df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 Success! GI Dataset saved to {SAVE_PATH}")

fetch_contrave_gi()

🚀 Scanning for GI signals in Contrave cohort...
📊 Found 1321 GI-related reports for Contrave.
✅ Progress: 1000 / 1321
✅ Progress: 1321 / 1321
🎉 Success! GI Dataset saved to /content/drive/MyDrive/IK/Datasets/Contrave_GI_Signals_2023_2025.parquet


In [ ]:
import requests
import pandas as pd
import time

from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION ---
API_KEY = "hmOpDFtToadeCdx1EtHk0TonEDIPeB20v7jiEsW4"
SAVE_PATH = "/content/drive/MyDrive/IK/Datasets/Contrave_GI_Signals_2014_2022.parquet"

# MedDRA Terms: Common vs. Serious GI Injury
GI_TERMS = [
    'NAUSEA', 'VOMITING', 'DIARRHOEA','CONSTIPATION', # Acute/Tolerability
    'GASTROPARESIS', 'INTESTINAL OBSTRUCTION',      # Serious/Structural
    'PANCREATITIS', 'GASTRIC STASIS', 'ILEUS'       # Serious/Functional
]

def fetch_contrave_gi():
    # Drug Query (Brand OR Generic Combo)
    drug_query = '(patient.drug.brand_name:"CONTRAVE" OR (patient.drug.openfda.substance_name:"NALTREXONE HYDROCHLORIDE" AND patient.drug.openfda.substance_name:"BUPROPION HYDROCHLORIDE"))'

    # Reaction Query (Standardized MedDRA PTs)
    reaction_query = " OR ".join([f'patient.reaction.reactionmeddrapt.exact:"{term}"' for term in GI_TERMS])

    # Combined Search String for 2023-2025
    full_search = f'{drug_query} AND ({reaction_query}) AND receivedate:[20140101 TO 20221231]'

    print(f"🚀 Scanning for GI signals in Contrave cohort...")

    base_url = f"https://api.fda.gov/drug/event.json?api_key={API_KEY}&search={full_search}"

    try:
        count_resp = requests.get(f"{base_url}&limit=1")
        total_records = count_resp.json().get('meta', {}).get('results', {}).get('total', 0)
        print(f"📊 Found {total_records} GI-related reports for Contrave.")

        if total_records == 0:
            print("💡 Observation: No GI signals found. Contrave appears highly tolerated in this dataset.")
            return
    except Exception as e:
        print(f"❌ Connection Error: {e}")
        return

    all_results = []
    skip = 0
    while skip < total_records:
        url = f"{base_url}&limit=1000&skip={skip}"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json().get('results', [])
            all_results.extend(data)
            print(f"✅ Progress: {len(all_results)} / {total_records}")
            skip += 1000
            time.sleep(0.2)
        else:
            break

    # Convert to DataFrame and Save
    df = pd.json_normalize(all_results)

    # Tag these records for later ML classification
    df['is_control_group'] = True
    df['drug_class'] = 'CNS_WEIGHT_LOSS'

    df.to_parquet(SAVE_PATH, index=False)
    print(f"🎉 Success! GI Dataset saved to {SAVE_PATH}")

fetch_contrave_gi()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Scanning for GI signals in Contrave cohort...
📊 Found 1389 GI-related reports for Contrave.
✅ Progress: 1000 / 1389
✅ Progress: 1389 / 1389
🎉 Success! GI Dataset saved to /content/drive/MyDrive/IK/Datasets/Contrave_GI_Signals_2014_2022.parquet


In [ ]:
import requests
import pandas as pd

# We search ONLY by Brand Name to ensure we catch every possible report
# Then we will manually count the GI terms in Python
url = f"https://api.fda.gov/drug/event.json?api_key={API_KEY}&search=patient.drug.brand_name:\"CONTRAVE\"+AND+receivedate:[20230101+TO+20251231]&limit=1000"

print("🔍 Pulling raw Contrave reports for manual signal audit...")

response = requests.get(url)
if response.status_code == 200:
    results = response.json().get('results', [])
    df_raw = pd.json_normalize(results)

    # Flatten the nested reactions list to see what's actually in there
    # This reaches deep into the JSON to find the MedDRA terms
    all_reactions = []
    for res in results:
        for reac in res.get('patient', {}).get('reaction', []):
            all_reactions.append(reac.get('reactionmeddrapt', '').upper())

    # Count the top side effects
    from collections import Counter
    counts = Counter(all_reactions)

    print("\n📊 Top 5 Actual Reported Side Effects for Contrave:")
    for term, count in counts.most_common(5):
        print(f"- {term}: {count} reports")
else:
    print(f"❌ API Error: {response.status_code}")

🔍 Pulling raw Contrave reports for manual signal audit...
❌ API Error: 403


In [ ]:
# Manually adding Phase 3 trial safety data as a benchmark
# Source: ACHIEVE-3 / ATTAIN-1 Trial Results (Published 2025/2026)
orforglipron_stats = {
    'Drug': 'Orforglipron (Phase 3)',
    'Total_Participants': 1698,
    'Muscle_Weakness_Rate': 0.02, # Approx 2% based on early clinical summaries
    'GI_Adverse_Events': 0.58,    # 58% reporting rate
    'Route': 'Oral (Small Molecule)'
}

print("✅ Orforglipron Benchmark Data added for 2026 Comparison.")

✅ Orforglipron Benchmark Data added for 2026 Comparison.
